In [1]:
from pathlib import Path
import duckdb

DATA = Path("data")
con = duckdb.connect()
con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")

print(con.execute("""
    SELECT
        CASE WHEN w = 0 THEN 'test (w=0)'
             WHEN w = 1 THEN 'valid (w=1)'
             ELSE 'train (w=2-25)' END AS split,
        COUNT(*) AS rows,
        COUNT(DISTINCT customer_id) AS customers,
        COUNT(DISTINCT article_id) AS articles,
        MIN(t_dat) AS first_day,
        MAX(t_dat) AS last_day
    FROM tx
    GROUP BY split
    ORDER BY first_day
""").df().to_string(index=False))

         split    rows  customers  articles  first_day   last_day
train (w=2-25) 1518983     143708     38083 2020-03-25 2020-09-08
   valid (w=1)   51364      14471     10282 2020-09-09 2020-09-15
    test (w=0)   47660      13637      9713 2020-09-16 2020-09-22


In [2]:
for name, week in [("valid", 1), ("test", 0)]:
    out = (DATA / f"labels_{name}.parquet").as_posix()
    con.execute(f"""
        COPY (
            SELECT customer_id, LIST(DISTINCT article_id) AS bought
            FROM tx
            WHERE w = {week}
            GROUP BY customer_id
        ) TO '{out}' (FORMAT parquet)
    """)
    print(name, con.execute(f"""
        SELECT COUNT(*) AS customers,
               ROUND(AVG(LEN(bought)), 2) AS avg_items,
               MEDIAN(LEN(bought)) AS median_items,
               MAX(LEN(bought)) AS max_items
        FROM '{out}'
    """).df().to_string(index=False))

valid  customers  avg_items  median_items  max_items
     14471       3.16           2.0         38
test  customers  avg_items  median_items  max_items
     13637       3.12           2.0         41


In [3]:
for name, week in [("valid", 1), ("test", 0)]:
    print(name, con.execute(f"""
        WITH hist AS (
            SELECT DISTINCT customer_id FROM tx WHERE w > {week}
        )
        SELECT
            COUNT(*) AS buyers,
            COUNT(h.customer_id) AS with_history,
            COUNT(*) - COUNT(h.customer_id) AS no_history,
            ROUND(100.0 * COUNT(h.customer_id) / COUNT(*), 1) AS pct_with_history
        FROM '{(DATA / f"labels_{name}.parquet").as_posix()}' l
        LEFT JOIN hist h USING (customer_id)
    """).df().to_string(index=False))

valid  buyers  with_history  no_history  pct_with_history
  14471         11898        2573              82.2
test  buyers  with_history  no_history  pct_with_history
  13637         11290        2347              82.8


## Train, validation and test split

I split by time, never at random. A random split would let a model peek at future weeks, which can't happen in real life.

- Train: weeks 2 to 25 (25 Mar to 8 Sep 2020)
- Validation: week 1 (9 to 15 Sep)
- Test: week 0 (16 to 22 Sep)

I tune and compare everything on the validation week. I only score on the test week once, at the very end, after retraining on weeks 1 to 25. If I kept checking the test week while tuning, my final number would end up looking better than it really is.

The answer key for each week is the list of different items each customer bought. A typical buyer bought only 2 items, but the task asks for 12 guesses, so most guesses will miss even for a good model.

About 18% of the people I'm scoring had no purchases in my training weeks. With full two-year history that would have been about 8%, so my 26-week cut costs me roughly 10 points here. I'll report results for customers with and without history separately.